# Data Preparation and Preprocessing

To align the dataset with the proposed two-stage modelling framework, a series of targeted preprocessing steps are applied. These steps address the structural dependencies, multicollinearity, and data limitations identified during EDA.

## Outline:

### Target Variable Transformation

The target variable (**G3**) is prepared in two distinct formats:

- **Risk Model:** A binary indicator where G3 = 0 represents non-completion and G3 > 0 represents completion.
- **Performance Model:** The dataset is restricted to observations where G3 > 0, ensuring the regression model is not influenced by non-completion cases.
 

### Feature Engineering and Consolidation

- Highly correlated features are evaluated for consolidation to reduce redundancy whilst preserving predictive signal. Candidate transformations are assessed on the **Performance Dataset** using Spearman correlation with **G3**, and the most informative representation is retained.

  - **Parental Education:** Composite measures are constructed using the **maximum** (Pedu_max) and **average** (Pedu_avg) of **Medu** and **Fedu**. Correlation analysis shows that **maximum parental education** exhibits the strongest relationship with G3, and is therefore retained in place of the original features.

  - **Alcohol Consumption:** Multiple representations are evaluated, including a weighted weekly total $(5 \times \text{Dalc} + 2 \times \text{Walc})$, the **maximum** (alc_max), and the **average** (alc_avg). The **average alcohol consumption** demonstrates the strongest association with G3, and is retained as the primary feature.

- **Absences:** A log transformation $\log(x + 1)$ is applied to reduce the strong right-skew identified during EDA. This limits the influence of extreme values whilst preserving their relative signal.

### Multicollinearity and Leakage

Several features introduce redundancy or leakage and are handled accordingly:

- **G2 is excluded from all models:** 
  - For the Risk Model, G2 = 0 is a deterministic predictor of G3 = 0 (early withdrawal), introducing **target leakage**. 
  - For the Performance Model, **G2 is almost perfectly correlated with G3** (r = 0.96). Its inclusion would yield trivial predictions and mask the influence of other features, offering little additional explanatory value.  

 - **G1 Strategy:**
   - **Both** models are evaluated **with and without G1** to assess the predictive value of socio-environmental factors alone and the additional contribution of early academic performance.
 

### Handling Data Constraints

A key limitation identified during EDA is that **absence data is missing for non-completers**:

- The Risk Model **excludes absences**, as zero-values are ambiguous, representing either genuine zero absences or missing absence data for non-completers.
- The Performance Model retains absences, where it provides a meaningful measure of engagement.


### Scaling and Encoding

- **Categorical Encoding:**  
  - Binary features are mapped to {0,1}.  
  - Nominal features (e.g. Mjob, Fjob) are one-hot encoded, with first-category dropping to prevent multicollinearity. 
  - Ordinal features (e.g. failures) are retained as ranked numeric variables.

- **Standardisation:**  
   - **Min-Max scaling** is applied to numerical features, transforming values to the range [0,1]. This ensures consistency across variables and supports subsequent feature engineering during modelling. 
   - **G1 is included** for scaling to support distance-based models.
   - **G3 is excluded** from scaling to preserve interpretability and ensure final predictions remain on the original 0–20 scale.

These preprocessing decisions ensure that each model is trained on features that are both valid and informative for its respective task.

## Implementation

### Setup

#### Imports

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

#### Loading The Data

In [2]:
path = os.path.join("..", "data", "raw", "Math-Students.csv")

df = pd.read_csv(path)

### Data Splitting and Target Definitions

#### Risk Data:

In [3]:
df_risk = df.copy()
df_risk["is_non_completer"] = (df_risk["G3"] == 0).astype(int)

#### Performance Data:

In [4]:
df_perf = df[df["G3"] > 0].copy()

### Feature Engineering Validation

Following the strategy outlined in the roadmap, candidate transformations are assessed on the Performance Dataset ($G3 > 0$) using Spearman correlation with **G3** to identify the strongest predictive signals.

In [5]:
df_val = df_perf.copy()

df_val["Pedu_max"] = df_val[["Medu", "Fedu"]].max(axis=1)
df_val["Pedu_avg"] = df_val[["Medu", "Fedu"]].mean(axis=1)

df_val["alc_weighted"] = (df_val["Dalc"] * 5) + (df_val["Walc"] * 2)
df_val["alc_max"] = df_val[["Dalc", "Walc"]].max(axis=1)
df_val["alc_avg"] = df_val[["Dalc", "Walc"]].mean(axis=1)

edu_cols = ["G3", "Medu", "Fedu", "Pedu_max", "Pedu_avg"]
alc_cols = ["G3", "Dalc", "Walc", "alc_weighted", "alc_max", "alc_avg"]

edu_corr = df_val[edu_cols].corr(method="spearman")["G3"].drop("G3")
alc_corr = df_val[alc_cols].corr(method="spearman")["G3"].drop("G3")
                     
edu_corr_ranked = edu_corr.reindex(edu_corr.abs().sort_values(ascending=False).index)
alc_corr_ranked = alc_corr.reindex(alc_corr.abs().sort_values(ascending=False).index)                     

print("Parental Education vs G3:")
print(edu_corr_ranked)
print("\nAlcohol Consumption vs G3:")
print(alc_corr_ranked)

Parental Education vs G3:
Pedu_max    0.195341
Medu        0.168457
Pedu_avg    0.167841
Fedu        0.138806
Name: G3, dtype: float64

Alcohol Consumption vs G3:
alc_avg        -0.187368
alc_max        -0.184554
alc_weighted   -0.183373
Walc           -0.182951
Dalc           -0.139688
Name: G3, dtype: float64


Based on the observed correlation magnitudes, **Pedu_max** and **alc_avg** are identified as the optimal representations and are retained in place of original features.

### Feature Engineering Implementation and Leakage Prevention

The validated transformations and structural constraints are implemented via a single function. This step consolidates features, applies log-scaling to absences, and removes redundant or leaking variables (G2) to ensure model integrity. G1 is optionally removed, allowing models to be evaluated both with and without this feature.

In [6]:
def apply_feature_engineering(df_input, model_type="performance", include_g1=True):
    
    data = df_input.copy()
    
    data["Pedu_max"] = data[["Medu", "Fedu"]].max(axis=1)
    data["alc_avg"] = data[["Dalc", "Walc"]].mean(axis=1)
    data["absences_log"] = np.log1p(data["absences"])
    
    to_drop = ["Medu", "Fedu", "Dalc", "Walc", "absences", "G2"]
    
    if model_type == "risk":
        to_drop.extend(["absences_log", "G3"])
        
    if not include_g1:
        to_drop.append("G1")
        
    data.drop(columns=to_drop, inplace=True, errors="ignore")
    
    return data

### Encoding and Scaling Implementation

A single function encodes categorical variables and scales numeric features to finalise the dataset for modeling. Encoding is applied dynamically based on feature type, and G3 is excluded from scaling to preserve interpretability.

In [7]:
def apply_formatting(df_input, scaler=None):
    data = df_input.copy()
    
    binary_cols = [col for col in data.select_dtypes(include=["object"]).columns 
                       if data[col].nunique() == 2]
    
    for col in binary_cols:
        
        data[col] = data[col].apply(lambda x: 1 if x in ["yes", "MS", "U", "T", "LE3", "M"] else 0)
        

    cat_cols = [col for col in data.select_dtypes(include=["object"]).columns 
                        if data[col].nunique() > 2]
    
    data = pd.get_dummies(data, columns=cat_cols, drop_first=True, dtype=int)
    
    cols_to_scale = [col for col in data.select_dtypes(include=[np.number]).columns
        if data[col].nunique() > 2 and col != "G3"]
    
    if scaler is None:
        scaler = MinMaxScaler()
        data[cols_to_scale] = scaler.fit_transform(data[cols_to_scale])
        return data, scaler
    else:
        data[cols_to_scale] = scaler.transform(data[cols_to_scale])
        return data


### Full Preprocessing Pipeline

Feature engineering and formatting steps are combined into a single preprocessing pipeline:

In [8]:
def full_preprocessing_pipeline(raw_df, model_type="performance", include_g1=True, scaler=None):
    
    df_engineered = apply_feature_engineering(raw_df, model_type=model_type, include_g1=include_g1)
    
    return apply_formatting(df_engineered, scaler=scaler)

## Preprocessing Utility Function

The preprocessing pipeline is implemented as a reusable utility function stored in a separate script (**preprocessing_utils.py**) and imported directly into the modelling notebook. This ensures consistent preprocessing and allows scaling transformations to be applied **after** train-test splitting to prevent data leakage.

### Dataset Configurations

The pipeline supports four dataset configurations, allowing the contribution of early academic performance (G1) to be assessed separately from environmental and behavioural factors:

* **Risk (Socio-Env):** Predicts non-completion risk using environmental and behavioural features only. This dataset is primarily used for feature importance analysis and feature engineering.
* **Risk (Full):** Predicts non-completion risk using both socio-environmental factors and early academic performance (G1).
* **Performance (Socio-Env):** Predicts final grade (G3) using environmental and behavioural features only. This dataset is used to investigate the independent contribution of non-academic factors and guide further feature engineering.
* **Performance (Full):** Predicts final grade (G3) using both socio-environmental factors and G1.